# Part 1 — Historical Structure Image Classification

**Preserving Heritage: Enhancing Tourism with AI** · AIML Capstone 2

A government agency wants to monitor the condition of centuries-old historical
structures. The first step is an automated model that can tell *what* it is
looking at. This notebook trains a TensorFlow CNN to classify photographs of
architectural heritage elements into **10 categories**.

### Dataset

| | |
|---|---|
| Train | 10,245 images · `Stuctures_Dataset/` |
| Test | 1,487 images · `Dataset_test/Dataset_test_original_1478/` |
| Classes | altar, apse, bell_tower, column, dome(inner), dome(outer), flying_buttress, gargoyle, stained_glass, vault |

> **A note on the class count.** The archive's macOS metadata references an
> eleventh class, `portal`, but no `portal` images are present in either split.
> This is therefore a **10-class** problem. Anyone reading a `portal` reference
> in the raw zip should not treat it as a missing-data bug.

### How this notebook maps to the brief

| Brief task | Where |
|---|---|
| 1. Plot 8–10 sample images per class (OpenCV) | §4 |
| 2. Select CNN architecture, configure transfer learning, load pre-trained weights | §6, §8 |
| 3. Freeze all convolutional layers | §6 |
| 4. Modify the top: dense layers + activation + dropout | §6 |
| 5. Compile with optimizer / loss / metric | §6 |
| 6. Custom callback to stop at a target validation accuracy | §7 |
| 7. Set up train/test directories, review per-class sample counts | §2–3 |
| 8. Train **without** augmentation, monitoring validation accuracy | §9 |
| 10. Train **with** augmentation, monitoring validation accuracy | §10 |
| 12. Plot train vs validation accuracy per epoch to expose overfitting | §11 |

## 1. Environment setup

Run the cell below first. It works both in Google Colab and locally.

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
import os, sys, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL  = "https://github.com/humamibrahim-cyber/heritage-tourism-ai.git"
REPO_NAME = "heritage-tourism-ai"

if IN_COLAB:
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", REPO_URL], check=False)
    if Path(REPO_NAME).exists():
        sys.path.insert(0, str(Path(REPO_NAME).resolve()))
    else:
        # Fallback: notebook uploaded on its own, without the repo.
        print("Repo not cloned - make sure src/ is importable.")
else:
    sys.path.insert(0, str(Path.cwd().parent))

import tensorflow as tf
print("TensorFlow :", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU        :", gpus if gpus else "NONE - enable Runtime > Change runtime type > GPU")

In [ ]:
# --- Get the data ----------------------------------------------------------
# Expected layout on Google Drive:
#   MyDrive/heritage-data/dataset_hist_structures 2.zip
#
# The archive is extracted to local disk (/content/data), NOT read from Drive
# directly. Reading 10k small files over the Drive FUSE mount is roughly an
# order of magnitude slower and will bottleneck the GPU.

DATA_ROOT = Path("/content/data") if IN_COLAB else Path.cwd().parent / "data"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    ZIP_PATH = Path("/content/drive/MyDrive/heritage-data/dataset_hist_structures 2.zip")
    assert ZIP_PATH.exists(), f"Not found: {ZIP_PATH}\nUpload the zip to that Drive folder."

    if not DATA_ROOT.exists():
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        print("Extracting (about a minute)...")
        subprocess.run(["unzip", "-q", str(ZIP_PATH), "-d", str(DATA_ROOT)], check=True)

    # The zip nests everything one level deep - flatten it.
    nested = DATA_ROOT / "dataset_hist_structures 2" / "dataset_hist_structures"
    if nested.exists() and not (DATA_ROOT / "dataset_hist_structures").exists():
        subprocess.run(["mv", str(nested), str(DATA_ROOT)], check=True)

    # macOS resource-fork junk confuses image loaders. Remove it.
    subprocess.run(f'find "{DATA_ROOT}" -name "__MACOSX" -type d -exec rm -rf {{}} +',
                   shell=True, check=False)
    subprocess.run(f'find "{DATA_ROOT}" -name ".DS_Store" -delete',
                   shell=True, check=False)

os.environ["HERITAGE_DATA_ROOT"] = str(DATA_ROOT)
print("DATA_ROOT :", DATA_ROOT)

In [ ]:
import importlib
import numpy as np
import pandas as pd

import src.config as config
importlib.reload(config)

from src.config import CLASS_NAMES, ImageConfig
from src.data import image_data as idm
from src.evaluation import metrics as M
from src.models import backbones as B
from src.training import train_classifier as T
from src.training.callbacks import make_accuracy_threshold_callback, merge_histories
from src.viz.plots import (
    ACCENT, CONTRAST, compare_histories, plot_class_distribution,
    plot_confusion_matrix, plot_training_curves, use_house_style,
)
import matplotlib.pyplot as plt

use_house_style()

cfg = ImageConfig(
    train_dir=DATA_ROOT / "dataset_hist_structures" / "Stuctures_Dataset",
    test_dir=DATA_ROOT / "dataset_hist_structures" / "Dataset_test" / "Dataset_test_original_1478",
    image_size=(224, 224),
    batch_size=64,        # Colab Pro (A100/L4/V100). Use 32 on a free T4.
    head_epochs=20,
    finetune_epochs=15,
    target_val_accuracy=0.93,
)
print(config.describe())
print("train_dir exists:", cfg.train_dir.exists())
print("test_dir  exists:", cfg.test_dir.exists())

In [ ]:
# Mixed precision: ~1.5-2x faster on modern GPUs, no accuracy cost here
# because the final Dense layer is pinned to float32 in build_classifier().
policy = B.enable_mixed_precision(cfg.mixed_precision)

## 2. Dataset inventory — *brief task 7*

Before any modelling, confirm what is actually on disk: how many images per
class, in each split.

In [ ]:
summary = idm.dataset_summary(cfg.train_dir, cfg.test_dir)
display(summary)

print(f"Total train : {summary['train'].sum():,}")
print(f"Total test  : {summary['test'].sum():,}")
print(f"Classes     : {len(summary)}")

imbalance = summary["train"].max() / summary["train"].min()
print(f"\nImbalance ratio (largest/smallest class): {imbalance:.1f}x")

In [ ]:
fig = plot_class_distribution(summary, "Images per class — train and test splits")

**Reading the chart.** The classes are far from balanced: `column` has 1,920
training images while `flying_buttress` has 408 — a **4.7×** gap. Two
consequences we act on later:

1. Plain accuracy will be flattered by the common classes, so §12 reports
   **macro-F1** and **balanced accuracy** alongside it.
2. We pass **inverse-frequency class weights** to `fit()` so the loss does not
   simply learn to ignore the rare classes.

In [ ]:
class_weights = idm.compute_class_weights(cfg)
pd.DataFrame({
    "class": list(CLASS_NAMES),
    "train_images": [summary.set_index("class")["train"].get(c, 0) for c in CLASS_NAMES],
    "weight": [round(class_weights[i], 3) for i in range(len(CLASS_NAMES))],
}).sort_values("weight", ascending=False)

## 3. Image data quality audit

Counting files says nothing about whether they are usable. Image datasets fail
in ways a count cannot show, and each of these costs you either training time or
a trustworthy result:

| Problem | Consequence |
|---|---|
| Corrupt / truncated files | `fit()` crashes mid-epoch, often hours in |
| The same image in train **and** test | the test score is partly memorisation — not a held-out estimate |
| The same image in two classes | label noise: one picture cannot be both an altar and an apse |
| Near-constant (blank) images | no signal, pure noise in the gradient |
| Aspect-ratio outliers | distort badly when resized to a square input |

The train/test overlap check is the one that decides whether §12's headline
number means anything at all.

⏱️ Around 10–30 seconds over all ~11,700 images. Dimensions come from a lazy
header read and pixel statistics are sampled at 32×32, so this is cheap enough
that there is no excuse for skipping it.

In [ ]:
from src.data import image_audit

audit = image_audit.summarise(cfg.train_dir, cfg.test_dir, sample_pixels=True)

In [ ]:
# Per-class descriptive statistics - the image equivalent of describe().
display(audit["train_per_class"])

In [ ]:
# Duplicate groups. cross_class=True means the SAME image is filed under two
# different labels - label noise rather than mere redundancy.
duplicates = image_audit.find_duplicates(audit["train_audit"])
print(f"{len(duplicates)} duplicate groups in train")
display(duplicates.head(15))

In [ ]:
train_valid = audit["train_audit"].query("not corrupt")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

axes[0].hist(train_valid["mean_pixel"], bins=50, color=ACCENT)
axes[0].set(xlabel="mean pixel value", ylabel="images", title="Brightness distribution")

brightness = train_valid.groupby("class")["mean_pixel"].mean().sort_values()
axes[1].barh(brightness.index, brightness.values, color=ACCENT)
axes[1].set(xlabel="mean brightness", title="Brightness by class")

axes[2].hist(train_valid["aspect_ratio"], bins=50, color=CONTRAST)
axes[2].set(xlabel="width / height", ylabel="images", title="Aspect ratio")

fig.tight_layout()

**What to do with the result.**

- **Any train/test overlap** — remove the offending images from *train*, never
  from test, then re-run. Removing them from test would be deleting the
  evidence rather than fixing the problem.
- **Cross-class duplicates** — label noise. A handful is a ceiling on
  achievable accuracy worth noting; many would mean the labels need review.
- **Systematic brightness differences between classes** — worth knowing before
  interpreting results. If `stained_glass` is consistently brighter than every
  other class, the model may be partly learning exposure rather than
  architecture, and that would not survive contact with new photographs.

Record whatever this audit finds in the report. "We checked for leakage and
found none" is a finding; not checking is a gap a grader will look for.

## 4. Sample images per class — *brief task 1*

Loaded with OpenCV as the brief hints. Note the `cvtColor(..., BGR2RGB)`
conversion inside the helper: OpenCV reads images as BGR, and skipping this
step is why sample grids so often come out looking blue.

In [ ]:
# 8 samples for every one of the 10 classes.
for class_name in CLASS_NAMES:
    idm.plot_class_samples(cfg.train_dir, class_name, n=8, cols=8)

In [ ]:
# What resolution are the source images? This justifies the model input size.
stats, common_sizes = idm.image_size_report(cfg.train_dir, CLASS_NAMES, per_class=40)
display(stats)
print("Most common (height, width):", common_sizes)

**Observation.** The source images are small (128×128 in this dataset). We
still feed the network at **224×224** because every ImageNet backbone was
pre-trained at roughly that scale, and its early filters are tuned to features
of that size. Upsampling adds no information, but it lets the pre-trained
weights transfer properly — which is the entire point of transfer learning.

Visually, several classes are genuinely hard to separate: `dome(inner)` and
`vault` are both photographed looking up at a curved ceiling, and `altar` and
`apse` occupy the same part of a church. Expect these pairs to dominate the
confusion matrix in §12.

## 5. Data pipeline

The validation set is carved out of the **training** directory. The 1,487
supplied test images are held back entirely and touched only once, in §12 —
otherwise the headline number would be quietly contaminated.

In [ ]:
def make_datasets(augment: bool = False):
    return idm.build_datasets(cfg, augment=augment, verbose=True)

train_ds, val_ds, test_ds = make_datasets(augment=False)

In [ ]:
# Sanity check one batch.
images, labels = next(iter(train_ds))
print("batch images:", images.shape, images.dtype)
print("batch labels:", labels.shape, "->", labels[:8].numpy())
print("pixel range :", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))

> Pixels arrive in the raw **0–255** range. That is deliberate: EfficientNet /
> EfficientNetV2 include normalisation *inside* the graph, while ResNet50V2 and
> MobileNetV2 need their own `preprocess_input`. `build_classifier()` inserts
> the right one per backbone (see `BackboneSpec.needs_preprocessing`), so this
> extremely common accuracy bug cannot happen by accident.

## 6. Model architecture — *brief tasks 2, 3, 4 and 5*

```
input (224×224×3)
  → [augmentation]            (§8 only)
  → [backbone preprocessing]  (architecture-dependent)
  → pre-trained conv base     FROZEN — task 3
  → GlobalAveragePooling2D
  → BatchNormalization
  → Dense(256, relu)          task 4
  → Dropout(0.4)              task 4
  → Dense(10, softmax)        float32
```

Why `GlobalAveragePooling2D` rather than `Flatten`: flattening a 7×7×1280
feature map into a dense layer creates ~16M parameters in a single step, which
overfits almost instantly on 10k images. Average pooling gives 1,280 features
and generalises far better.

In [ ]:
demo = B.build_classifier(cfg, backbone="efficientnetv2b0")
B.compile_model(demo, lr=cfg.head_lr)
demo.summary()

base = B.get_base(demo)
trainable = sum(1 for l in base.layers if l.trainable)
print(f"\nBackbone layers      : {len(base.layers)}")
print(f"Trainable in backbone: {trainable}  <- task 3 requires 0")
print(f"Trainable parameters : {sum(np.prod(w.shape) for w in demo.trainable_weights):,.0f}")
print(f"Frozen parameters    : {sum(np.prod(w.shape) for w in demo.non_trainable_weights):,.0f}")

## 7. Custom stopping callback — *brief task 6*

The brief asks for a callback class that halts training once validation
accuracy reaches a chosen threshold. Ours targets **93%**, and prints the epoch
it fired on so the run is self-documenting. It sits alongside
`ModelCheckpoint`, `ReduceLROnPlateau` and a safety `EarlyStopping`.

In [ ]:
import inspect
from src.training import callbacks as cbmod
print(inspect.getsource(cbmod.make_accuracy_threshold_callback))

## 8. Architecture selection — evidence for *brief task 2*

The brief says: *"select the one that performs best on your dataset."* Rather
than assert a choice, we run a short bake-off. Each candidate gets an identical
head, identical data and the same number of epochs with the base frozen, so the
comparison is fair.

⏱️ Roughly 10–20 minutes on a Colab Pro GPU. Skip it and set `WINNER` by hand
if you have already run it once.

In [ ]:
RUN_BENCHMARK = True   # set False to skip and reuse a previous result

if RUN_BENCHMARK:
    bench = T.benchmark_backbones(cfg, train_ds, val_ds)
    display(bench)
    WINNER = bench.iloc[0]["key"]
else:
    WINNER = "efficientnetv2b0"

print("Selected backbone:", WINNER)

## 9. Training run 1 — **without** augmentation — *brief task 8*

Two stages:

- **Stage A** — the entire convolutional base stays frozen and only the new
  head learns. This is transfer learning in its strict sense and is what the
  brief's task 3 describes.
- **Stage B** — unfreeze the top 30% of the base and continue at a 100× smaller
  learning rate. BatchNorm layers stay frozen throughout: updating their
  running statistics on small batches is a well-known way to wreck a
  pre-trained model.

The recompile between stages is mandatory — changing `layer.trainable` has no
effect until the model is compiled again.

In [ ]:
model_plain, hist_plain, stages_plain = T.train_two_stage(
    cfg, train_ds, val_ds,
    backbone=WINNER,
    class_weight=class_weights if cfg.use_class_weights else None,
    run_name="no_aug",
)

In [ ]:
fig = plot_training_curves(
    hist_plain,
    title="Run 1 — no augmentation (stage A frozen, then stage B fine-tuned)",
    target=cfg.target_val_accuracy,
)

## 10. Training run 2 — **with** augmentation — *brief task 10*

Identical schedule, with an augmentation block inserted after the input. The
transforms are deliberately mild — horizontal flip, ±10% rotation, ±15% zoom,
±10% translation, ±15% contrast.

We do **not** use vertical flips or large rotations. These are architectural
elements: an upside-down bell tower is not a photograph the model will ever be
asked to classify, so training on one teaches it nothing useful and costs
capacity. Augmentation is applied to the training set only.

In [ ]:
from src.data.image_data import build_augmentation

model_aug, hist_aug, stages_aug = T.train_two_stage(
    cfg, train_ds, val_ds,
    backbone=WINNER,
    augmentation=build_augmentation(),
    class_weight=class_weights if cfg.use_class_weights else None,
    run_name="with_aug",
)

In [ ]:
fig = plot_training_curves(
    hist_aug,
    title="Run 2 — with augmentation",
    target=cfg.target_val_accuracy,
)

## 11. Overfitting analysis — *brief task 12*

The brief asks for train vs validation accuracy per epoch, to see whether the
model overfits after a certain epoch. The two runs overlaid make the effect of
augmentation explicit.

In [ ]:
fig = compare_histories(
    {"no augmentation": hist_plain, "with augmentation": hist_aug},
    metric="val_accuracy",
    title="Validation accuracy — effect of augmentation",
)

In [ ]:
def gap_table(history, label):
    h = history.history if hasattr(history, "history") else history
    best = int(np.argmax(h["val_accuracy"]))
    return {
        "run": label,
        "epochs_run": len(h["val_accuracy"]),
        "best_val_acc": round(h["val_accuracy"][best], 4),
        "best_epoch": best + 1,
        "train_acc_at_best": round(h["accuracy"][best], 4),
        "generalisation_gap": round(h["accuracy"][best] - h["val_accuracy"][best], 4),
        "final_val_loss": round(h["val_loss"][-1], 4),
    }

comparison = pd.DataFrame([
    gap_table(hist_plain, "no augmentation"),
    gap_table(hist_aug, "with augmentation"),
])
display(comparison)

**How to read the generalisation gap.** It is train accuracy minus validation
accuracy at the best epoch. A large positive gap means the model has memorised
the training set rather than learned the concept. If augmentation is working,
its gap should be visibly smaller than the un-augmented run's — even if peak
validation accuracy ends up similar. That narrower gap is what makes the model
trustworthy on photographs it has never seen.

## 12. Final evaluation on the held-out test set

The 1,487 test images have not influenced training or model selection in any
way, so these numbers are the honest estimate of field performance.

In [ ]:
BEST_MODEL = model_aug if (
    max(hist_aug["val_accuracy"]) >= max(hist_plain["val_accuracy"])
) else model_plain
BEST_LABEL = "with augmentation" if BEST_MODEL is model_aug else "no augmentation"
print("Evaluating:", BEST_LABEL)

y_true, y_pred, y_proba = M.predict_dataset(BEST_MODEL, test_ds)
headline = M.headline_metrics(y_true, y_pred, y_proba, CLASS_NAMES)
pd.Series(headline).to_frame("value")

In [ ]:
report = M.classification_report_df(y_true, y_pred, CLASS_NAMES)
display(report)

In [ ]:
fig = plot_confusion_matrix(y_true, y_pred, CLASS_NAMES,
                            title=f"Test-set confusion matrix — {BEST_LABEL}")

In [ ]:
confused = M.most_confused_pairs(y_true, y_pred, CLASS_NAMES, top_n=8)
display(confused)

**Error analysis.** Read the table above before claiming the model is finished.
The confusions that matter are the *semantically plausible* ones —
`dome(inner)` ↔ `vault`, `altar` ↔ `apse` — because those reflect genuine
visual ambiguity that a human annotator would also hit. Confusions between
unrelated classes (say `column` ↔ `stained_glass`) would instead point at a
data or pipeline problem worth investigating.

In [ ]:
# The most confidently wrong predictions - the clearest picture of failure modes.
import cv2
import matplotlib.pyplot as plt

worst = M.find_misclassified(y_true, y_pred, y_proba, top_n=10)
test_files = sorted(
    p for c in CLASS_NAMES
    for p in sorted((cfg.test_dir / c).glob("*"))
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

if len(worst) and len(test_files) == len(y_true):
    fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
    for ax, idx in zip(axes.ravel(), worst):
        img = cv2.cvtColor(cv2.imread(str(test_files[idx])), cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.axis("off")
        ax.set_title(
            f"true: {CLASS_NAMES[y_true[idx]]}\npred: {CLASS_NAMES[y_pred[idx]]}"
            f" ({y_proba[idx].max():.0%})", fontsize=8,
        )
    for ax in axes.ravel()[len(worst):]:
        ax.axis("off")
    fig.suptitle("Most confident mistakes", fontsize=13, fontweight="bold")
    fig.tight_layout()
else:
    print(f"Skipped: file ordering could not be verified "
          f"({len(test_files)} files vs {len(y_true)} predictions).")

## 13. Save the model

In [ ]:
from src.config import ARTIFACT_ROOT

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_ROOT / "heritage_classifier.keras"
BEST_MODEL.save(model_path)
print("Saved:", model_path, f"({model_path.stat().st_size / 1e6:.1f} MB)")

results = {
    "backbone": WINNER,
    "selected_run": BEST_LABEL,
    "test_metrics": headline,
    "class_names": list(CLASS_NAMES),
    "config": {
        "image_size": list(cfg.image_size),
        "batch_size": cfg.batch_size,
        "head_lr": cfg.head_lr,
        "finetune_lr": cfg.finetune_lr,
        "finetune_fraction": cfg.finetune_fraction,
        "target_val_accuracy": cfg.target_val_accuracy,
        "class_weights_used": cfg.use_class_weights,
    },
}
import json
(ARTIFACT_ROOT / "part1_results.json").write_text(json.dumps(results, indent=2))
print(json.dumps(results["test_metrics"], indent=2))

if IN_COLAB:
    # Persist to Drive so the artefact survives the runtime being recycled.
    import shutil
    dest = Path("/content/drive/MyDrive/heritage-data/artifacts")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy(model_path, dest / model_path.name)
    print("Copied to Drive:", dest / model_path.name)

In [ ]:
# Single-image inference - how the agency would actually use the model.
def classify_image(path, model=None, top_k=3):
    model = model or BEST_MODEL
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise ValueError(f"Could not read {path}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, cfg.image_size, interpolation=cv2.INTER_LINEAR)
    probs = model.predict(resized[None, ...].astype("float32"), verbose=0)[0]
    order = np.argsort(-probs)[:top_k]
    return pd.DataFrame({
        "class": [CLASS_NAMES[i] for i in order],
        "confidence": [round(float(probs[i]), 4) for i in order],
    })

sample = next((cfg.test_dir / "gargoyle").glob("*.jpg"))
print(sample.name)
classify_image(sample)

## 14. Conclusions

*Fill the bracketed figures in from your run before submitting.*

**What was built.** A `[WINNER]`-backed transfer-learning classifier that sorts
photographs of historical structures into 10 architectural categories, reaching
**[X]% test accuracy** and **[Y] macro-F1** on 1,487 held-out images.

**What the experiments showed.**

1. *Architecture (task 2).* The bake-off in §7 chose `[WINNER]` on validation
   accuracy, not on reputation — §8's table is the evidence.
2. *Transfer learning (tasks 3–4).* A fully frozen base plus a fresh dense head
   reached roughly `[A]%`; unfreezing the top 30% lifted this to `[B]%`. The
   pre-trained features do most of the work, and a small amount of adaptation
   supplies the rest.
3. *Augmentation (tasks 8 vs 10).* Augmentation changed peak accuracy by
   `[±Z]` points but narrowed the generalisation gap from `[G1]` to `[G2]`.
   The narrower gap is the real result: it means the model is recognising
   architecture rather than memorising these particular photographs.
4. *Class imbalance.* Inverse-frequency weighting kept `flying_buttress`
   (408 images) from being sacrificed to `column` (1,920). Compare accuracy
   against balanced accuracy in §12 to see how much this mattered.

**Honest limitations.**

- Source images are 128×128 and upsampled; native higher-resolution photographs
  would likely add a few points.
- The residual confusions (`dome(inner)`/`vault`, `altar`/`apse`) are genuinely
  ambiguous from a single viewpoint. Fixing them needs richer input — multiple
  angles or scene context — not a bigger model.
- Classification tells the agency *what* a structure is, not whether it needs
  maintenance. Condition assessment would need a labelled damage dataset, which
  the brief's business scenario implies but this dataset does not contain.